# XAUUSD SMC Monte Carlo Trading Algorithm

Run the first cell once to install dependencies, then run the main algorithm cell. The output is formatted as a decision dashboard with a clear decision, trade plan, confirmation trigger, conditional entry, SL, TP1, TP2, and invalidation level.


In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/rongardF/tvdatafeed.git pandas numpy -q


In [ ]:
import pandas as pd
import numpy as np
import time
from datetime import datetime
from zoneinfo import ZoneInfo
from tvDatafeed import TvDatafeed, Interval

SYMBOL = "XAUUSD"
EXCHANGE = "OANDA"

BUFFER = 0.20
RR_CHOICES = [4, 3, 2]

NORMAL_MIN_MC = 50
SCALP_MIN_MC = 45

BARS_5M = 500
BARS_15M = 300
BARS_1H = 300
BARS_4H = 300

MC_SIMULATIONS = 500
MC_STEPS = 12

SWING_LOOKBACK = 3
SMC_LOOKBACK = 80
FVG_LOOKBACK = 25
DISPLACEMENT_ATR_MULT = 0.65
MIN_SMC_SCORE = 2.0
CONSOLIDATION_MIN_SMC_SCORE = 2.5

tv = TvDatafeed()

STATE_PARAMS = {
    "LAMINAR_BULL": {"drift": 0.18, "vol": 0.90},
    "LAMINAR_BEAR": {"drift": -0.18, "vol": 0.90},
    "TURBULENT_BULL": {"drift": 0.08, "vol": 1.60},
    "TURBULENT_BEAR": {"drift": -0.08, "vol": 1.60},
    "TRANSITION_BULL": {"drift": 0.10, "vol": 1.20},
    "TRANSITION_BEAR": {"drift": -0.10, "vol": 1.20},
    "CHOP": {"drift": 0.00, "vol": 1.10},
}

def ema(s, n):
    return s.ewm(span=n, adjust=False).mean()

def atr(df, n=14):
    pc = df["close"].shift(1)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - pc).abs(),
        (df["low"] - pc).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(n).mean()

def rsi(s, n=14):
    delta = s.diff()
    gain = delta.clip(lower=0).rolling(n).mean()
    loss = (-delta.clip(upper=0)).rolling(n).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def macd(df):
    fast = ema(df["close"], 12)
    slow = ema(df["close"], 26)
    line = fast - slow
    signal = ema(line, 9)
    hist = line - signal
    return line, signal, hist

def bollinger(df, n=20, k=2):
    mid = df["close"].rolling(n).mean()
    std = df["close"].rolling(n).std()
    upper = mid + k * std
    lower = mid - k * std
    width = (upper - lower) / mid.replace(0, np.nan)
    return upper, mid, lower, width

def donchian(df, n=20):
    return df["high"].rolling(n).max(), df["low"].rolling(n).min()

def laplacian(s):
    return s - 2 * s.shift(1) + s.shift(2)

def swing_points(df, lookback=SWING_LOOKBACK):
    highs = df["high"]
    lows = df["low"]
    swing_high = pd.Series(True, index=df.index)
    swing_low = pd.Series(True, index=df.index)

    for i in range(1, lookback + 1):
        swing_high &= highs > highs.shift(i)
        swing_high &= highs >= highs.shift(-i)
        swing_low &= lows < lows.shift(i)
        swing_low &= lows <= lows.shift(-i)

    swing_high = swing_high.fillna(False)
    swing_low = swing_low.fillna(False)
    return swing_high, swing_low

def fetch_tv(interval, bars):
    df = tv.get_hist(symbol=SYMBOL, exchange=EXCHANGE, interval=interval, n_bars=bars)
    if df is None or df.empty:
        raise ValueError("TradingView OANDA feed returned no data")

    df = df.copy()
    df.columns = [str(c).lower() for c in df.columns]

    if "volume" not in df.columns:
        df["volume"] = 0

    return df[["open", "high", "low", "close", "volume"]].dropna()

def prepare_df(df):
    df = df.copy()

    df["EMA9"] = ema(df["close"], 9)
    df["EMA21"] = ema(df["close"], 21)
    df["EMA50"] = ema(df["close"], 50)
    df["ATR"] = atr(df)
    df["RSI"] = rsi(df["close"])
    df["LAP"] = laplacian(df["close"])

    _, _, macd_hist = macd(df)
    df["MACD_HIST"] = macd_hist
    df["MACD_HIST_CHANGE"] = df["MACD_HIST"].diff()

    _, bb_mid, _, bb_width = bollinger(df)
    df["BB_MID"] = bb_mid
    df["BB_WIDTH"] = bb_width
    df["BB_WIDTH_MA"] = df["BB_WIDTH"].rolling(50).mean()

    don_high, don_low = donchian(df)
    df["DONCHIAN_HIGH"] = don_high
    df["DONCHIAN_LOW"] = don_low

    df["LAMINAR"] = abs(df["EMA9"] - df["EMA21"]) / df["ATR"].replace(0, np.nan)
    df["ATR_MA"] = df["ATR"].rolling(50).mean()
    df["ATR_RATIO"] = df["ATR"] / df["ATR_MA"].replace(0, np.nan)

    df["BODY"] = (df["close"] - df["open"]).abs()
    df["RANGE"] = (df["high"] - df["low"]).replace(0, np.nan)
    df["BODY_RATIO"] = df["BODY"] / df["RANGE"]

    required = [
        "open", "high", "low", "close", "volume",
        "EMA9", "EMA21", "EMA50", "ATR", "RSI", "LAP",
        "MACD_HIST", "MACD_HIST_CHANGE", "BB_MID", "BB_WIDTH",
        "BB_WIDTH_MA", "DONCHIAN_HIGH", "DONCHIAN_LOW",
        "LAMINAR", "ATR_MA", "ATR_RATIO", "BODY", "RANGE", "BODY_RATIO"
    ]

    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=required)

    df["SWING_HIGH"], df["SWING_LOW"] = swing_points(df)

    df["BULL_FVG_LOW"] = np.where(df["low"] > df["high"].shift(2), df["high"].shift(2), np.nan)
    df["BULL_FVG_HIGH"] = np.where(df["low"] > df["high"].shift(2), df["low"], np.nan)
    df["BEAR_FVG_LOW"] = np.where(df["high"] < df["low"].shift(2), df["high"], np.nan)
    df["BEAR_FVG_HIGH"] = np.where(df["high"] < df["low"].shift(2), df["low"].shift(2), np.nan)

    return df

def get_bias(df):
    row = df.iloc[-1]

    bull = 0
    bear = 0

    bull += int(row["EMA9"] > row["EMA21"])
    bear += int(row["EMA9"] < row["EMA21"])

    bull += int(row["close"] > row["EMA50"])
    bear += int(row["close"] < row["EMA50"])

    bull += int(row["MACD_HIST"] > 0)
    bear += int(row["MACD_HIST"] < 0)

    bull += int(row["RSI"] > 50)
    bear += int(row["RSI"] < 50)

    if bull >= 3:
        return "BULLISH"
    elif bear >= 3:
        return "BEARISH"
    return "NEUTRAL"

def get_context_4h(df):
    row = df.iloc[-1]

    if row["ATR_RATIO"] > 2.50:
        return "TOO_VOLATILE"

    if row["BB_WIDTH"] < row["BB_WIDTH_MA"] or row["ATR_RATIO"] < 0.60:
        return "CONSOLIDATING"

    return "TRADABLE"

def get_ny_time_and_session():
    ny = datetime.now(ZoneInfo("America/New_York"))
    ist = datetime.now(ZoneInfo("Asia/Kolkata"))

    h_ny = ny.hour + ny.minute / 60
    h_ist = ist.hour + ist.minute / 60

    if 18.5 <= h_ist <= 20:
        return ny, 1.35, "SCALP_WINDOW"

    if 7 <= h_ny < 11:
        return ny, 1.25, "NY_POWER"
    elif 11 <= h_ny < 13:
        return ny, 0.90, "NY_MIDDAY"
    elif 13 <= h_ny < 16:
        return ny, 1.05, "NY_AFTERNOON"

    return ny, 0.0, "NO_TRADE_SESSION"

def get_zones(df_15m, df_1h, price):
    recent_15 = df_15m.tail(48)
    recent_1h = df_1h.tail(48)

    atr_15 = float(df_15m.iloc[-1]["ATR"])
    atr_1h = float(df_1h.iloc[-1]["ATR"])

    d_low_15 = float(recent_15["low"].min())
    d_high_15 = d_low_15 + atr_15 * 0.50

    s_high_15 = float(recent_15["high"].max())
    s_low_15 = s_high_15 - atr_15 * 0.50

    d_low_1h = float(recent_1h["low"].min())
    d_high_1h = d_low_1h + atr_1h * 0.35

    s_high_1h = float(recent_1h["high"].max())
    s_low_1h = s_high_1h - atr_1h * 0.35

    demands = [("15M", d_low_15, d_high_15), ("1H", d_low_1h, d_high_1h)]
    supplies = [("15M", s_low_15, s_high_15), ("1H", s_low_1h, s_high_1h)]

    valid_demands = [z for z in demands if z[2] < price]
    valid_supplies = [z for z in supplies if z[1] > price]

    demand = max(valid_demands, key=lambda z: z[2]) if valid_demands else demands[0]
    supply = min(valid_supplies, key=lambda z: z[1]) if valid_supplies else supplies[0]

    d_tf, d_low, d_high = demand
    s_tf, s_low, s_high = supply

    d_width = max(d_high - d_low, 0.01)
    s_width = max(s_high - s_low, 0.01)

    return {
        "demand_tf": d_tf,
        "d_low": d_low,
        "d_high": d_high,
        "d_buf_low": d_low - BUFFER * d_width,
        "d_buf_high": d_high + BUFFER * d_width,
        "supply_tf": s_tf,
        "s_low": s_low,
        "s_high": s_high,
        "s_buf_low": s_low - BUFFER * s_width,
        "s_buf_high": s_high + BUFFER * s_width
    }

def get_last_swing_levels(df, before_index=-1, lookback=SMC_LOOKBACK):
    recent = df.iloc[:before_index].tail(lookback)
    swing_highs = recent[recent["SWING_HIGH"]]
    swing_lows = recent[recent["SWING_LOW"]]

    last_high = None if swing_highs.empty else float(swing_highs.iloc[-1]["high"])
    last_low = None if swing_lows.empty else float(swing_lows.iloc[-1]["low"])
    return last_high, last_low

def get_recent_fvg(df, direction, lookback=FVG_LOOKBACK):
    recent = df.tail(lookback)

    if direction == "BUY":
        gaps = recent.dropna(subset=["BULL_FVG_LOW", "BULL_FVG_HIGH"])
        if gaps.empty:
            return None
        row = gaps.iloc[-1]
        return {
            "type": "BULLISH_FVG",
            "low": float(row["BULL_FVG_LOW"]),
            "high": float(row["BULL_FVG_HIGH"]),
            "mid": float((row["BULL_FVG_LOW"] + row["BULL_FVG_HIGH"]) / 2)
        }

    gaps = recent.dropna(subset=["BEAR_FVG_LOW", "BEAR_FVG_HIGH"])
    if gaps.empty:
        return None
    row = gaps.iloc[-1]
    return {
        "type": "BEARISH_FVG",
        "low": float(row["BEAR_FVG_LOW"]),
        "high": float(row["BEAR_FVG_HIGH"]),
        "mid": float((row["BEAR_FVG_LOW"] + row["BEAR_FVG_HIGH"]) / 2)
    }

def get_smc_profile(df, direction, zones):
    row = df.iloc[-1]
    prev = df.iloc[-2]
    price = float(row["close"])
    atr_now = float(row["ATR"])
    last_swing_high, last_swing_low = get_last_swing_levels(df)

    bullish_displacement = bool(
        row["close"] > row["open"]
        and row["BODY"] >= atr_now * DISPLACEMENT_ATR_MULT
        and row["BODY_RATIO"] >= 0.55
    )
    bearish_displacement = bool(
        row["close"] < row["open"]
        and row["BODY"] >= atr_now * DISPLACEMENT_ATR_MULT
        and row["BODY_RATIO"] >= 0.55
    )

    bull_sweep = bool(
        last_swing_low is not None
        and prev["low"] < last_swing_low
        and price > last_swing_low
    )
    bear_sweep = bool(
        last_swing_high is not None
        and prev["high"] > last_swing_high
        and price < last_swing_high
    )

    bull_mss = bool(
        last_swing_high is not None
        and price > last_swing_high
    )
    bear_mss = bool(
        last_swing_low is not None
        and price < last_swing_low
    )

    bull_fvg = get_recent_fvg(df, "BUY")
    bear_fvg = get_recent_fvg(df, "SELL")

    if direction == "BUY":
        fvg = bull_fvg
        fvg_near = bool(fvg is not None and fvg["low"] <= price <= fvg["high"] + atr_now * 0.35)
        score = sum([
            bull_sweep,
            bull_mss,
            bullish_displacement,
            fvg is not None,
            fvg_near
        ])
        refined_entry = fvg["mid"] if fvg is not None and price >= fvg["mid"] else price
        structural_sl = min(
            zones["d_buf_low"],
            last_swing_low if last_swing_low is not None else zones["d_buf_low"]
        )
        structural_sl -= atr_now * 0.10
        valid = bool(score >= MIN_SMC_SCORE)

        return {
            "score": float(score),
            "valid": valid,
            "sweep": bull_sweep,
            "mss": bull_mss,
            "choch": bull_mss and bull_sweep,
            "displacement": bullish_displacement,
            "fvg": fvg,
            "fvg_near": fvg_near,
            "entry": float(refined_entry),
            "sl": float(structural_sl),
            "last_swing_high": last_swing_high,
            "last_swing_low": last_swing_low,
            "notes": "BUY needs sell-side sweep, bullish shift, displacement or FVG quality"
        }

    fvg = bear_fvg
    fvg_near = bool(fvg is not None and fvg["low"] - atr_now * 0.35 <= price <= fvg["high"])
    score = sum([
        bear_sweep,
        bear_mss,
        bearish_displacement,
        fvg is not None,
        fvg_near
    ])
    refined_entry = fvg["mid"] if fvg is not None and price <= fvg["mid"] else price
    structural_sl = max(
        zones["s_buf_high"],
        last_swing_high if last_swing_high is not None else zones["s_buf_high"]
    )
    structural_sl += atr_now * 0.10
    valid = bool(score >= MIN_SMC_SCORE)

    return {
        "score": float(score),
        "valid": valid,
        "sweep": bear_sweep,
        "mss": bear_mss,
        "choch": bear_mss and bear_sweep,
        "displacement": bearish_displacement,
        "fvg": fvg,
        "fvg_near": fvg_near,
        "entry": float(refined_entry),
        "sl": float(structural_sl),
        "last_swing_high": last_swing_high,
        "last_swing_low": last_swing_low,
        "notes": "SELL needs buy-side sweep, bearish shift, displacement or FVG quality"
    }

def get_market_state(row):
    atr_ratio = float(row["ATR_RATIO"])
    laminar = float(row["LAMINAR"]) > 0.05

    ema_bull = row["EMA9"] > row["EMA21"]
    ema_bear = row["EMA9"] < row["EMA21"]

    lap_bull = row["LAP"] > 0
    lap_bear = row["LAP"] < 0

    if laminar and ema_bull and lap_bull:
        return "LAMINAR_BULL"
    if laminar and ema_bear and lap_bear:
        return "LAMINAR_BEAR"

    if atr_ratio > 1.80 and ema_bull:
        return "TURBULENT_BULL"
    if atr_ratio > 1.80 and ema_bear:
        return "TURBULENT_BEAR"

    if ema_bull or lap_bull:
        return "TRANSITION_BULL"
    if ema_bear or lap_bear:
        return "TRANSITION_BEAR"

    return "CHOP"

def monte_carlo_probability(entry, sl, tp, state, atr_now, simulations=MC_SIMULATIONS, steps=MC_STEPS):
    params = STATE_PARAMS.get(state, STATE_PARAMS["CHOP"])

    drift = params["drift"] * atr_now
    vol = params["vol"] * atr_now

    tp_hits = 0
    sl_hits = 0

    direction = "BUY" if tp > entry else "SELL"

    for _ in range(simulations):
        sim_price = entry

        for _ in range(steps):
            move = np.random.normal(loc=drift, scale=vol)
            sim_price += move

            if direction == "BUY":
                if sim_price >= tp:
                    tp_hits += 1
                    break
                if sim_price <= sl:
                    sl_hits += 1
                    break

            else:
                if sim_price <= tp:
                    tp_hits += 1
                    break
                if sim_price >= sl:
                    sl_hits += 1
                    break

    total = tp_hits + sl_hits

    if total == 0:
        return 0.0

    return round((tp_hits / total) * 100, 2)

def choose_best_rr(direction, entry, sl, zones, state, atr_now, smc_score):
    best = None

    for rr in RR_CHOICES:
        risk = abs(entry - sl)

        if risk <= 0:
            continue

        if direction == "BUY":
            tp = entry + rr * risk
            structure_valid = tp <= zones["s_buf_low"]
        else:
            tp = entry - rr * risk
            structure_valid = tp >= zones["d_buf_high"]

        if not structure_valid:
            continue

        mc_prob = monte_carlo_probability(entry, sl, tp, state, atr_now)
        smc_bonus = min(smc_score, 5.0) * 0.05
        expectancy = (mc_prob / 100.0) * rr - (1 - mc_prob / 100.0) + smc_bonus

        if best is None or expectancy > best["expectancy"]:
            best = {
                "rr": rr,
                "tp": tp,
                "mc_prob": mc_prob,
                "expectancy": expectancy
            }

    return best

def build_exit_rule(direction, entry, sl, tp, smc):
    risk = abs(entry - sl)

    if risk <= 0:
        return "WAIT"

    one_r = entry + risk if direction == "BUY" else entry - risk

    return (
        f"EXIT AT TP OR SL | MOVE SL TO BREAKEVEN AFTER 1R ({one_r:.2f}) | "
        "IF OPPOSITE MSS/CHOCH APPEARS, REDUCE OR EXIT"
    )

def yes_no(value):
    return "YES" if bool(value) else "NO"

def format_price(value):
    return "N/A" if value is None else f"{value:.2f}"

def format_percent(value):
    return "N/A" if value is None else f"{value:.2f}%"

def get_smc_read(smc, min_smc_score):
    confirmations = []

    if smc["sweep"]:
        confirmations.append("liquidity sweep")
    if smc["mss"]:
        confirmations.append("market structure shift")
    if smc["choch"]:
        confirmations.append("CHoCH")
    if smc["displacement"]:
        confirmations.append("displacement")
    if smc["fvg_near"]:
        confirmations.append("price near FVG")

    if not confirmations:
        confirmations_text = "no major SMC confirmation yet"
    else:
        confirmations_text = ", ".join(confirmations)

    status = "PASS" if smc["score"] >= min_smc_score else "WAIT"
    return f"{status} - SMC score {smc['score']:.1f}/{min_smc_score:.1f}; {confirmations_text}"

def get_no_trade_reason(
    session_mult, session_name, atr_ok, context_4h, bias_1h,
    setup_long, setup_short, smc_long_ok, smc_short_ok,
    buy_score, sell_score, smc_long, smc_short, min_smc_score
):
    if session_mult == 0:
        return f"No trade: current time is outside the approved trading sessions ({session_name})."
    if not atr_ok:
        return "No trade: current volatility is outside the safe operating range."
    if context_4h == "TOO_VOLATILE":
        return "No trade: 4H market is too volatile for this model."
    if bias_1h == "NEUTRAL":
        return "No trade: 1H trend is neutral, so direction is not clear enough."

    if buy_score >= sell_score:
        if bias_1h != "BULLISH":
            return "No trade: buy side is stronger, but 1H bias is not bullish."
        if not setup_long:
            return "No trade: bullish bias exists, but there is no clean demand, sweep, or pullback setup."
        if not smc_long_ok:
            return f"No trade: bullish SMC confirmation is weak ({smc_long['score']:.1f}/{min_smc_score:.1f})."
        return "No trade: bullish setup exists, but RR or Monte Carlo probability did not pass."

    if bias_1h != "BEARISH":
        return "No trade: sell side is stronger, but 1H bias is not bearish."
    if not setup_short:
        return "No trade: bearish bias exists, but there is no clean supply, sweep, or pullback setup."
    if not smc_short_ok:
        return f"No trade: bearish SMC confirmation is weak ({smc_short['score']:.1f}/{min_smc_score:.1f})."
    return "No trade: bearish setup exists, but RR or Monte Carlo probability did not pass."

def build_confirmation_plan(signal, setup_direction, price, zones, active_smc, atr_now, rr_used=None):
    direction = signal if signal in ["BUY", "SELL"] else setup_direction.replace(" WATCH", "")
    direction = "BUY" if direction not in ["BUY", "SELL"] and setup_direction.startswith("BUY") else direction
    direction = "SELL" if direction not in ["BUY", "SELL"] else direction

    fvg = active_smc["fvg"]

    if direction == "BUY":
        zone_low = zones["d_buf_low"]
        zone_high = zones["d_buf_high"]
        trigger = active_smc["last_swing_high"]
        if trigger is None or trigger <= price:
            trigger = price + atr_now * 0.25

        entry = trigger
        sl = active_smc["sl"]
        risk = max(entry - sl, 0.01)
        tp_1r = entry + risk
        tp_2r = entry + risk * 2
        structure_target = zones["s_buf_low"]
        final_tp = min(tp_2r, structure_target) if structure_target > entry else tp_2r

        location = f"demand zone {zone_low:.2f}-{zone_high:.2f}"
        if fvg is not None:
            location += f" or bullish FVG {fvg['low']:.2f}-{fvg['high']:.2f}"

        if signal == "BUY":
            summary = "Confirmed BUY is active."
            trigger_text = f"Entry is already valid around {price:.2f}; avoid chasing if price runs far above entry."
        else:
            summary = "Conditional BUY watch."
            trigger_text = (
                f"Enter only if price holds/rebounds from {location} and then closes above {trigger:.2f}."
            )

        invalidation = f"Invalid if price closes below {sl:.2f}."
        targets = f"TP1 {tp_1r:.2f} | TP2 {final_tp:.2f}"

    else:
        zone_low = zones["s_buf_low"]
        zone_high = zones["s_buf_high"]
        trigger = active_smc["last_swing_low"]
        if trigger is None or trigger >= price:
            trigger = price - atr_now * 0.25

        entry = trigger
        sl = active_smc["sl"]
        risk = max(sl - entry, 0.01)
        tp_1r = entry - risk
        tp_2r = entry - risk * 2
        structure_target = zones["d_buf_high"]
        final_tp = max(tp_2r, structure_target) if structure_target < entry else tp_2r

        location = f"supply zone {zone_low:.2f}-{zone_high:.2f}"
        if fvg is not None:
            location += f" or bearish FVG {fvg['low']:.2f}-{fvg['high']:.2f}"

        if signal == "SELL":
            summary = "Confirmed SELL is active."
            trigger_text = f"Entry is already valid around {price:.2f}; avoid chasing if price falls far below entry."
        else:
            summary = "Conditional SELL watch."
            trigger_text = (
                f"Enter only if price rejects {location} and then closes below {trigger:.2f}."
            )

        invalidation = f"Invalid if price closes above {sl:.2f}."
        targets = f"TP1 {tp_1r:.2f} | TP2 {final_tp:.2f}"

    return {
        "summary": summary,
        "trigger": trigger_text,
        "entry": round(entry, 2),
        "sl": round(sl, 2),
        "tp1": round(tp_1r, 2),
        "tp2": round(final_tp, 2),
        "invalidation": invalidation,
        "targets": targets,
    }

def print_signal(sig):
    print("\n" + "=" * 58)
    print("XAUUSD DECISION DASHBOARD")
    print("=" * 58)
    print(f"DECISION: {sig['DECISION']}")
    print(f"ACTION:   {sig['ACTION']}")
    print(f"REASON:   {sig['REASON']}")
    print("-" * 58)
    print(f"PRICE: {format_price(sig['LIVE_PRICE'])} | TIME: {sig['LIVE_TIME']} NY | SESSION: {sig['SESSION']}")
    print(f"MARKET: 1H {sig['BIAS_1H']} | 4H {sig['BIAS_4H']} | CONTEXT {sig['CONTEXT_4H']} | STATE {sig['STATE']}")
    print(f"SETUP: {sig['SETUP_DIRECTION']} | BUY SCORE {sig['BUY_SCORE']} | SELL SCORE {sig['SELL_SCORE']}")
    print(f"SMC: {sig['SMC_READ']}")
    print("-" * 58)

    if sig["SIGNAL"] in ["BUY", "SELL"]:
        print("TRADE PLAN")
        print(f"ENTRY: {format_price(sig['ENTRY'])}")
        print(f"STOP:  {format_price(sig['SL'])}")
        print(f"TARGET:{format_price(sig['TP'])}")
        print(f"RR: {sig['RR']} | MONTE CARLO: {format_percent(sig['SUCCESS_PERCENT'])}")
        print(f"EXIT: {sig['EXIT']}")
    else:
        print("TRADE PLAN")
        print("WAIT. No valid entry yet.")
        print("Next step: wait for stronger alignment before taking risk.")

    print("-" * 58)
    print("CONFIRMATION PLAN")
    print(sig["CONFIRMATION_SUMMARY"])
    print(sig["CONFIRMATION_TRIGGER"])
    print(f"Conditional entry: {format_price(sig['CONFIRM_ENTRY'])}")
    print(f"Confirmation SL:  {format_price(sig['CONFIRM_SL'])}")
    print(f"Confirmation TP:  TP1 {format_price(sig['CONFIRM_TP1'])} | TP2 {format_price(sig['CONFIRM_TP2'])}")
    print(sig["CONFIRM_INVALIDATION"])

    print("-" * 58)
    print(f"STRUCTURE: Demand {sig['DEMAND_TF']} | Supply {sig['SUPPLY_TF']} | FVG {sig['SMC_FVG']}")
    print(
        f"SMC CHECKS: Sweep {sig['SMC_SWEEP']} | MSS {sig['SMC_MSS']} | "
        f"CHoCH {sig['SMC_CHOCH']} | Displacement {sig['SMC_DISPLACEMENT']}"
    )
    print("=" * 58 + "\n")

def signal_once():
    df_5m = prepare_df(fetch_tv(Interval.in_5_minute, BARS_5M))
    df_15m = prepare_df(fetch_tv(Interval.in_15_minute, BARS_15M))
    df_1h = prepare_df(fetch_tv(Interval.in_1_hour, BARS_1H))
    df_4h = prepare_df(fetch_tv(Interval.in_4_hour, BARS_4H))

    bias_1h = get_bias(df_1h)
    bias_4h = get_bias(df_4h)
    context_4h = get_context_4h(df_4h)

    row = df_5m.iloc[-1]
    prev = df_5m.iloc[-2]

    candle_time = str(df_5m.index[-1])
    price = float(row["close"])
    atr_now = float(row["ATR"])

    ny_time, session_mult, session_name = get_ny_time_and_session()

    zones = get_zones(df_15m, df_1h, price)

    near_demand = zones["d_buf_low"] <= price <= zones["d_buf_high"]
    near_supply = zones["s_buf_low"] <= price <= zones["s_buf_high"]

    bull_reclaim = bool(prev["low"] <= prev["DONCHIAN_LOW"] and price > row["DONCHIAN_LOW"])
    bear_reject = bool(prev["high"] >= prev["DONCHIAN_HIGH"] and price < row["DONCHIAN_HIGH"])

    atr_ok = bool(0.50 <= float(row["ATR_RATIO"]) <= 2.50)

    macd_bull = bool(row["MACD_HIST"] > 0 or row["MACD_HIST_CHANGE"] > 0)
    macd_bear = bool(row["MACD_HIST"] < 0 or row["MACD_HIST_CHANGE"] < 0)

    rsi_buy_ok = bool(32 <= float(row["RSI"]) <= 75)
    rsi_sell_ok = bool(25 <= float(row["RSI"]) <= 68)

    ema_bull = bool(row["EMA9"] > row["EMA21"])
    ema_bear = bool(row["EMA9"] < row["EMA21"])

    lap_bull = bool(row["LAP"] > 0)
    lap_bear = bool(row["LAP"] < 0)

    bb_bull = bool(price > row["BB_MID"])
    bb_bear = bool(price < row["BB_MID"])

    laminar_ok = bool(row["LAMINAR"] > 0.035)

    tube_width = max(zones["s_buf_low"] - zones["d_buf_high"], 0.01)
    distance_to_supply = max(zones["s_buf_low"] - price, 0.01)
    distance_to_demand = max(price - zones["d_buf_high"], 0.01)

    velocity = abs(float(df_5m["close"].diff().iloc[-1]))
    k_laminar = max(float(row["LAMINAR"]), 0.01)

    graetz_long = (velocity * tube_width ** 2) / (k_laminar * distance_to_supply)
    graetz_short = (velocity * tube_width ** 2) / (k_laminar * distance_to_demand)

    pullback_long = bool(
        bias_1h == "BULLISH"
        and price > row["EMA21"]
        and price <= row["EMA9"] + atr_now * 0.60
    )

    pullback_short = bool(
        bias_1h == "BEARISH"
        and price < row["EMA21"]
        and price >= row["EMA9"] - atr_now * 0.60
    )

    smc_long = get_smc_profile(df_5m, "BUY", zones)
    smc_short = get_smc_profile(df_5m, "SELL", zones)

    min_smc_score = CONSOLIDATION_MIN_SMC_SCORE if context_4h == "CONSOLIDATING" else MIN_SMC_SCORE

    setup_long = near_demand or bull_reclaim or pullback_long or smc_long["sweep"]
    setup_short = near_supply or bear_reject or pullback_short or smc_short["sweep"]

    smc_long_ok = bool(smc_long["score"] >= min_smc_score)
    smc_short_ok = bool(smc_short["score"] >= min_smc_score)

    buy_score = sum([
        ema_bull, lap_bull, laminar_ok, setup_long,
        graetz_long > graetz_short, macd_bull,
        rsi_buy_ok, atr_ok, bb_bull, bias_1h == "BULLISH",
        smc_long_ok, smc_long["choch"], smc_long["displacement"], smc_long["fvg_near"]
    ])

    sell_score = sum([
        ema_bear, lap_bear, laminar_ok, setup_short,
        graetz_short > graetz_long, macd_bear,
        rsi_sell_ok, atr_ok, bb_bear, bias_1h == "BEARISH",
        smc_short_ok, smc_short["choch"], smc_short["displacement"], smc_short["fvg_near"]
    ])

    if bias_4h == "BULLISH":
        buy_score += 0.5
    elif bias_4h == "BEARISH":
        sell_score += 0.5

    buy_score *= session_mult
    sell_score *= session_mult

    state = get_market_state(row)

    signal = "NO TRADE"
    success = None
    entry = None
    sl = None
    tp = None
    rr_used = None
    exit_rule = "WAIT"
    smc_used = None

    min_mc = SCALP_MIN_MC if session_name == "SCALP_WINDOW" else NORMAL_MIN_MC

    if session_mult != 0 and atr_ok and context_4h != "TOO_VOLATILE":

        if bias_1h == "BULLISH" and setup_long and smc_long_ok and buy_score >= sell_score:
            test_entry = min(price, smc_long["entry"])
            test_sl = smc_long["sl"]

            if test_entry > test_sl:
                best = choose_best_rr("BUY", test_entry, test_sl, zones, state, atr_now, smc_long["score"])

                if best is not None and best["mc_prob"] >= min_mc:
                    signal = "BUY"
                    success = best["mc_prob"]
                    entry = test_entry
                    sl = test_sl
                    tp = best["tp"]
                    rr_used = best["rr"]
                    smc_used = smc_long
                    exit_rule = build_exit_rule("BUY", entry, sl, tp, smc_long)

        elif bias_1h == "BEARISH" and setup_short and smc_short_ok and sell_score >= buy_score:
            test_entry = max(price, smc_short["entry"])
            test_sl = smc_short["sl"]

            if test_entry < test_sl:
                best = choose_best_rr("SELL", test_entry, test_sl, zones, state, atr_now, smc_short["score"])

                if best is not None and best["mc_prob"] >= min_mc:
                    signal = "SELL"
                    success = best["mc_prob"]
                    entry = test_entry
                    sl = test_sl
                    tp = best["tp"]
                    rr_used = best["rr"]
                    smc_used = smc_short
                    exit_rule = build_exit_rule("SELL", entry, sl, tp, smc_short)

    active_smc = smc_used
    if active_smc is None:
        active_smc = smc_long if buy_score >= sell_score else smc_short

    fvg = active_smc["fvg"]
    setup_direction = "BUY WATCH" if buy_score >= sell_score else "SELL WATCH"

    if signal == "BUY":
        decision = "TAKE BUY"
        action = "Valid long setup. Use the trade plan below."
        reason = "Bullish bias, SMC confirmation, RR, and Monte Carlo filter all passed."
        setup_direction = "BUY"
    elif signal == "SELL":
        decision = "TAKE SELL"
        action = "Valid short setup. Use the trade plan below."
        reason = "Bearish bias, SMC confirmation, RR, and Monte Carlo filter all passed."
        setup_direction = "SELL"
    else:
        decision = "NO TRADE"
        action = "Wait. Conditions are not clean enough."
        reason = get_no_trade_reason(
            session_mult, session_name, atr_ok, context_4h, bias_1h,
            setup_long, setup_short, smc_long_ok, smc_short_ok,
            buy_score, sell_score, smc_long, smc_short, min_smc_score
        )

    smc_read = get_smc_read(active_smc, min_smc_score)
    confirmation = build_confirmation_plan(
        signal, setup_direction, price, zones, active_smc, atr_now, rr_used
    )

    return {
        "LIVE_TIME": ny_time.strftime("%Y-%m-%d %H:%M:%S"),
        "CANDLE_TIME": candle_time,
        "LIVE_PRICE": round(price, 2),
        "SIGNAL": signal,
        "DECISION": decision,
        "ACTION": action,
        "REASON": reason,
        "SESSION": session_name,
        "SETUP_DIRECTION": setup_direction,
        "BUY_SCORE": round(buy_score, 2),
        "SELL_SCORE": round(sell_score, 2),
        "CONFIRMATION_SUMMARY": confirmation["summary"],
        "CONFIRMATION_TRIGGER": confirmation["trigger"],
        "CONFIRM_ENTRY": confirmation["entry"],
        "CONFIRM_SL": confirmation["sl"],
        "CONFIRM_TP1": confirmation["tp1"],
        "CONFIRM_TP2": confirmation["tp2"],
        "CONFIRM_INVALIDATION": confirmation["invalidation"],
        "BIAS_1H": bias_1h,
        "BIAS_4H": bias_4h,
        "CONTEXT_4H": context_4h,
        "STATE": state,
        "SUCCESS_PERCENT": success,
        "RR": None if rr_used is None else f"1:{rr_used}",
        "ENTRY": None if entry is None else round(entry, 2),
        "SL": None if sl is None else round(sl, 2),
        "TP": None if tp is None else round(tp, 2),
        "DEMAND_TF": zones["demand_tf"],
        "SUPPLY_TF": zones["supply_tf"],
        "SMC_SCORE": active_smc["score"],
        "SMC_READ": smc_read,
        "SMC_SWEEP": active_smc["sweep"],
        "SMC_MSS": active_smc["mss"],
        "SMC_CHOCH": active_smc["choch"],
        "SMC_DISPLACEMENT": active_smc["displacement"],
        "SMC_FVG": None if fvg is None else f"{fvg['type']} {round(fvg['low'], 2)}-{round(fvg['high'], 2)}",
        "EXIT": exit_rule
    }

last_candle_time = None
first_run = True

while True:
    try:
        sig = signal_once()

        if first_run or sig["CANDLE_TIME"] != last_candle_time:
            first_run = False
            last_candle_time = sig["CANDLE_TIME"]
            print_signal(sig)

        time.sleep(20)

    except Exception as e:
        print("\nERROR:", e)
        time.sleep(60)
